In [2]:
# 2. Importamos las librerías necesarias
import dlt
from dlt.sources.helpers import requests
from datetime import datetime
import time

In [8]:
@dlt.source
def chess_source(players: list = None):
    if players is None:
        players = ['magnuscarlsen', 'rpragchess']
    # Un recurso para obtener las URLs de los archivos mensuales del jugador
    @dlt.resource
    def player_games_urls():
        base_url = "https://api.chess.com/pub/player/{player}/games/archives"
        for player in players:
            response = requests.get(base_url.format(player=player))
            response.raise_for_status()
            archives = response.json().get('archives', [])
            for url in archives:
                # 'dlt' añade metadatos útiles. Aquí extraemos el mes/año de la URL
                yield {
                    'player': player,
                    'url': url,
                    'month': url.split('/')[-1] # Ej: '2024/01'
                }
    # Devolvemos el/los recursos que componen la fuente
    return player_games_urls

In [14]:
@dlt.resource(
    primary_key='game_id', # Asumimos que 'game_id' es único por partida
    write_disposition='merge' # 'merge' nos permite actualizar o insertar. Para logs sería 'append'
)
def player_games_data(
    incremental_urls=dlt.sources.incremental('url') # 'url' es el campo que usaremos para trackear
):
    # Obtenemos el recurso 'player_games_urls' de nuestra fuente 'chess_source'
    source = chess_source()

    # Iteramos sobre cada URL que produce el recurso 'player_games_urls'
    for url_data in source.player_games_urls:
        # La magia de incremental: si esta URL ya fue procesada, la saltamos.
        # dlt.sources.incremental() gestiona esto automáticamente basado en el estado guardado.
        if incremental_urls.last_value is not None and url_data['url'] <= incremental_urls.last_value:
            print(f"Saltando URL ya procesada: {url_data['url']}")
            continue

        print(f"Procesando nueva URL: {url_data['url']} para el jugador {url_data['player']}")
        # Llamada a la API para obtener las partidas del mes
        response = requests.get(url_data['url'])
        response.raise_for_status()
        games = response.json().get('games', [])

        for game in games:
            # Enriquecer los datos con información de nuestro contexto
            # Usamos .get() para evitar KeyError si alguna clave no existe
            white_player_username = game.get('white', {}).get('username')
            black_player_username = game.get('black', {}).get('username')

            result = None
            if white_player_username == url_data['player']:
                result = game.get('white', {}).get('result')
            elif black_player_username == url_data['player']:
                result = game.get('black', {}).get('result')

            end_time_raw = game.get('end_time')
            end_time = None # Initialize end_time to None
            if end_time_raw is not None:
                try:
                    end_time = datetime.fromtimestamp(end_time_raw)
                except (TypeError, ValueError) as e:
                    # Log the error and set end_time to None
                    print(f"Error converting end_time '{end_time_raw}' for player {url_data['player']} in URL {url_data['url']}: {e}")
                    end_time = None

            yield {
                'game_id': game.get('url', '').split('/')[-1] if game.get('url') else None, # Extraemos un ID único
                'player': url_data['player'],
                'url': url_data['url'],
                'month': url_data['month'],
                'white_player': white_player_username,
                'black_player': black_player_username,
                'result': result,
                'pgn': game.get('pgn', 'N/A'), # Safely get 'pgn' with default value
                'time_control': game.get('time_control', 'N/A'), # Safely get 'time_control' with default value
                'end_time': end_time,
                'processed_at': datetime.now() # timestamp de cuando nuestro pipeline lo procesó
            }
        # Pequeña pausa para no saturar la API
        time.sleep(1)

In [17]:
# 1. Definimos el pipeline
pipeline = dlt.pipeline(
    pipeline_name='chess_incremental_pipeline', # Nombre único para este pipeline
    destination='duckdb', # Usamos DuckDB como destino
    dataset_name='chess_data' # Esquema o 'dataset' en nuestro destino
)

# 2. Ejecutamos la carga con nuestro recurso
# La primera ejecución cargará TODO.
print("--- PRIMERA EJECUCIÓN ---")
load_info_1 = pipeline.run(player_games_data())
print(load_info_1)

# Añadimos un chequeo de errores para la primera ejecución
if load_info_1.has_failed_jobs:
    print("ERROR: La primera ejecución del pipeline falló.")
    print(load_info_1.jobs_without_schema)
    # Optionally, you can raise an exception or handle the error further
else:
    print("La primera ejecución del pipeline completó exitosamente.")

# Veamos los datos cargados
print("\n--- DATOS EN LA TABLA (primera ejecución) ---")
with pipeline.sql_client() as client:
    # El nombre de la tabla por defecto es el nombre del recurso: 'player_games_data'
    res = client.execute_sql("SELECT player, month, COUNT(*) as game_count FROM player_games_data GROUP BY player, month ORDER BY month;")
    for row in res:
        print(row)

# 3. Simulamos una segunda ejecución (incremental)
# Dado que el estado recuerda la última URL procesada, solo cargará meses nuevos.
# Como no hay meses nuevos, no debería cargar nada.
print("\n--- SEGUNDA EJECUCIÓN (INCREMENTAL) ---")
load_info_2 = pipeline.run(player_games_data())
print(load_info_2)

# Añadimos un chequeo de errores para la segunda ejecución
if load_info_2.has_failed_jobs:
    print("ERROR: La segunda ejecución (incremental) del pipeline falló.")
    print(load_info_2.jobs_without_schema)
else:
    print("La segunda ejecución del pipeline completó exitosamente (posiblemente sin datos nuevos).")

--- PRIMERA EJECUCIÓN ---
Procesando nueva URL: https://api.chess.com/pub/player/magnuscarlsen/games/2014/12 para el jugador magnuscarlsen
Procesando nueva URL: https://api.chess.com/pub/player/magnuscarlsen/games/2016/06 para el jugador magnuscarlsen
Procesando nueva URL: https://api.chess.com/pub/player/magnuscarlsen/games/2016/08 para el jugador magnuscarlsen
Procesando nueva URL: https://api.chess.com/pub/player/magnuscarlsen/games/2016/10 para el jugador magnuscarlsen
Procesando nueva URL: https://api.chess.com/pub/player/magnuscarlsen/games/2017/02 para el jugador magnuscarlsen
Procesando nueva URL: https://api.chess.com/pub/player/magnuscarlsen/games/2017/03 para el jugador magnuscarlsen
Procesando nueva URL: https://api.chess.com/pub/player/magnuscarlsen/games/2017/05 para el jugador magnuscarlsen
Procesando nueva URL: https://api.chess.com/pub/player/magnuscarlsen/games/2017/10 para el jugador magnuscarlsen
Procesando nueva URL: https://api.chess.com/pub/player/magnuscarlsen/g

2026-04-09 13:36:25,095|[WARNING]|23346|140519649398784|dlt|__init__.py|_check_duplicate_cursor_threshold:680|Large number of records (201) sharing the same value of cursor field 'url' on resource 'player_games_data'. This can happen if the cursor field has a low resolution (e.g., only stores dates without times), causing many records to share the same cursor value. Consider using a cursor column with higher resolution to reduce the deduplication state size.


Procesando nueva URL: https://api.chess.com/pub/player/magnuscarlsen/games/2022/12 para el jugador magnuscarlsen


2026-04-09 13:36:26,617|[WARNING]|23346|140519649398784|dlt|__init__.py|_check_duplicate_cursor_threshold:680|Large number of records (201) sharing the same value of cursor field 'url' on resource 'player_games_data'. This can happen if the cursor field has a low resolution (e.g., only stores dates without times), causing many records to share the same cursor value. Consider using a cursor column with higher resolution to reduce the deduplication state size.


Procesando nueva URL: https://api.chess.com/pub/player/magnuscarlsen/games/2023/01 para el jugador magnuscarlsen


...

Procesando nueva URL: https://api.chess.com/pub/player/rpragchess/games/2021/06 para el jugador rpragchess
Procesando nueva URL: https://api.chess.com/pub/player/rpragchess/games/2021/07 para el jugador rpragchess
Procesando nueva URL: https://api.chess.com/pub/player/rpragchess/games/2021/08 para el jugador rpragchess
Procesando nueva URL: https://api.chess.com/pub/player/rpragchess/games/2021/09 para el jugador rpragchess


2026-04-09 13:38:55,028|[WARNING]|23346|140519649398784|dlt|__init__.py|_check_duplicate_cursor_threshold:680|Large number of records (201) sharing the same value of cursor field 'url' on resource 'player_games_data'. This can happen if the cursor field has a low resolution (e.g., only stores dates without times), causing many records to share the same cursor value. Consider using a cursor column with higher resolution to reduce the deduplication state size.


Procesando nueva URL: https://api.chess.com/pub/player/rpragchess/games/2021/10 para el jugador rpragchess
Procesando nueva URL: https://api.chess.com/pub/player/rpragchess/games/2021/11 para el jugador rpragchess
Procesando nueva URL: https://api.chess.com/pub/player/rpragchess/games/2021/12 para el jugador rpragchess
Procesando nueva URL: https://api.chess.com/pub/player/rpragchess/games/2022/02 para el jugador rpragchess
Procesando nueva URL: https://api.chess.com/pub/player/rpragchess/games/2022/03 para el jugador rpragchess
Procesando nueva URL: https://api.chess.com/pub/player/rpragchess/games/2022/04 para el jugador rpragchess
Procesando nueva URL: https://api.chess.com/pub/player/rpragchess/games/2022/05 para el jugador rpragchess
Procesando nueva URL: https://api.chess.com/pub/player/rpragchess/games/2022/06 para el jugador rpragchess
Procesando nueva URL: https://api.chess.com/pub/player/rpragchess/games/2022/07 para el jugador rpragchess
Procesando nueva URL: https://api.che

2026-04-09 13:39:49,894|[WARNING]|23346|140519649398784|dlt|__init__.py|_check_duplicate_cursor_threshold:680|Large number of records (201) sharing the same value of cursor field 'url' on resource 'player_games_data'. This can happen if the cursor field has a low resolution (e.g., only stores dates without times), causing many records to share the same cursor value. Consider using a cursor column with higher resolution to reduce the deduplication state size.


Procesando nueva URL: https://api.chess.com/pub/player/rpragchess/games/2025/06 para el jugador rpragchess
Procesando nueva URL: https://api.chess.com/pub/player/rpragchess/games/2025/07 para el jugador rpragchess
Procesando nueva URL: https://api.chess.com/pub/player/rpragchess/games/2025/08 para el jugador rpragchess
Procesando nueva URL: https://api.chess.com/pub/player/rpragchess/games/2025/09 para el jugador rpragchess
Procesando nueva URL: https://api.chess.com/pub/player/rpragchess/games/2025/10 para el jugador rpragchess
Procesando nueva URL: https://api.chess.com/pub/player/rpragchess/games/2025/11 para el jugador rpragchess
Procesando nueva URL: https://api.chess.com/pub/player/rpragchess/games/2025/12 para el jugador rpragchess
Procesando nueva URL: https://api.chess.com/pub/player/rpragchess/games/2026/01 para el jugador rpragchess
Procesando nueva URL: https://api.chess.com/pub/player/rpragchess/games/2026/02 para el jugador rpragchess
Pipeline chess_incremental_pipeline l

In [16]:
print("\n--- MUESTRA DE DATOS EN LA TABLA 'player_games_data' ---")
with pipeline.sql_client() as client:
    # Get the default schema name from the pipeline
    # Using pipeline.dataset_name directly to avoid SchemaNotFoundError if default_schema is not yet fully initialized
    schema_name = pipeline.dataset_name
    print(f"Checking tables in schema: {schema_name}")

    # Query the information schema to list all tables in the current dataset
    # DuckDB's information schema can be queried like this:
    res = client.execute_sql(f"SELECT table_name FROM information_schema.tables WHERE table_schema = '{schema_name}';")

    tables = [row[0] for row in res]
    if 'player_games_data' in tables:
        print("Table 'player_games_data' exists. Displaying first 5 rows:")
        res = client.execute_sql("SELECT * FROM player_games_data LIMIT 5;")
        for row in res:
            print(row)
    else:
        print(f"Table 'player_games_data' does not exist in schema '{schema_name}'. Found tables: {tables}")


--- MUESTRA DE DATOS EN LA TABLA 'player_games_data' ---
Checking tables in schema: chess_data
Table 'player_games_data' does not exist in schema 'chess_data'. Found tables: []


In [19]:
# Mostrar el estado guardado por el pipeline
print("\n--- ESTADO DEL PIPELINE (guardado en DuckDB) ---")
# El estado se guarda en una tabla interna llamada '_dlt_pipeline_state'
with pipeline.sql_client() as client:
    # La función 'get_storage_state' de dlt es más limpia para esto
    state = pipeline.state
    # Buscamos la información de nuestro recurso 'player_games_data'
    # Usar .get() para evitar KeyError si 'resources' no existe
    resource_state = state.get('resources', {}).get('player_games_data', {})
    print(f"Último valor de 'url' procesado: {resource_state.get('incremental', {}).get('url', {}).get('last_value')}")


--- ESTADO DEL PIPELINE (guardado en DuckDB) ---
Último valor de 'url' procesado: None
